In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import time
import joblib

from utils_ML import score_dataset
from utils_preproces import ols_fit, y_distribution_plot, corr_heatmap_plot, make_mi_scores, preprocess_data

from sklearn.ensemble import RandomForestRegressor, VotingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor, plot_importance
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


from sklearn.model_selection import train_test_split, StratifiedKFold, KFold


from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import QuantileTransformer

import optuna

import json
import joblib
from pathlib import Path


import random
from heapq import nsmallest

In [2]:
# Set Matplotlib defaults
plt.style.use("seaborn-v0_8-white")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

In [3]:
# Fix random state
random_state = 42

In [4]:
harps = pd.read_csv('harps_sample.csv')
apogee = pd.read_csv('apogee_sample.csv')
galah = pd.read_csv('galah_sample.csv')
gr8 = pd.read_csv('gr8_not_in_harps_sample.csv')

apogee = apogee[apogee.parallax > 0.]
galah = galah[(galah.parallax > 0.) ]

apogee['rel_parallax'] = apogee.parallax_error / apogee.parallax
apogee = apogee[apogee.rel_parallax < 0.1]

galah['rel_parallax'] = galah.parallax_error / galah.parallax
galah = galah[galah.rel_parallax < 0.1]


harps['rel_parallax'] = harps.parallax_error / harps.parallax
harps = harps[harps.rel_parallax < 0.1]

# Seelcting only giants
apogee = apogee[apogee.logg < 3]

# Selecting only dwarfs
galah = galah[galah.logg > 3]


apogee.isnull().sum() # 196 null value for P_gal
apogee.dropna(inplace = True)

galah.isnull().sum() # 1 null value for P_gal
galah.dropna(inplace = True)

#harps.isnull().sum() # No null values


galah = galah[(galah.zmax < 10) ]

apogee = apogee[(apogee.zmax < 10) ]
apogee.shape


print (f'harps contains {len(harps)} stars')
print (f'apogee contains {len(apogee)} stars')    
print (f'galah contains {len(galah)} stars')
print (f'gr8 contains {len(gr8)} stars')

harps contains 798 stars
apogee contains 45727 stars
galah contains 69449 stars
gr8 contains 1430 stars


In [5]:
def fit_trasform(X):
    #X['VW_lsr'] = np.sqrt(X['V_lsr']**2 + X['W_lsr']**2)
    #X['UW_lsr'] = np.sqrt(X['U_lsr']**2 + X['W_lsr']**2)
    #X['UV_lsr'] = np.sqrt(X['U_lsr']**2 + X['V_lsr']**2)
    X['UVW_lsr'] = np.sqrt(X['U_lsr']**2 + X['V_lsr']**2 + X['W_lsr']**2)
    epsilon = 1e-6
    #log features
    #X['Log_RV'] = np.log(X['RV'] + np.abs(X['RV'].min()))
    #X['Log_U_lsr'] = np.log(X['U_lsr'] + np.abs(X['U_lsr'].min()) + epsilon)
    #X['Log_V_lsr'] = np.log(X['V_lsr'] + np.abs(X['V_lsr'].min()) + epsilon)
    #X['abs_W_lsr'] =np.abs(X['W_lsr'])
    #X['abs_U_lsr'] =np.abs(X['U_lsr'])
    #X['Log_UW_lsr'] = np.log(X['UW_lsr'])
    #X['Log_UVW_lsr'] = np.log(X['UVW_lsr'])
    #X.drop(columns = ['U_lsr', 'W_lsr'], inplace = True)
    #X['R_mean'] = (X['rapo'] + X['rperi'])/2
    #X.drop(columns = ['rperi', 'rapo'], inplace = True)

    return X

apogee = fit_trasform(apogee)
harps = fit_trasform(harps)

In [6]:
PRETTY_LABELS = {
    "U_lsr":        r"$U_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "V_lsr":        r"$V_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "W_lsr":        r"$W_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UV_lsr":       r"$UV_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UW_lsr":       r"$UW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "VW_lsr":       r"$VW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UVW_lsr":      r"$UVW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "abs_U_lsr":    r"$|U|_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "abs_W_lsr":    r"$|W|_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "P_gal":        r"$P_{\mathrm{gal}}\,[\%]$",
    "rapo":         r"$R_{\mathrm{apo}}\,[\mathrm{kpc}]$",
    "rperi":        r"$R_{\mathrm{peri}}\,[\mathrm{kpc}]$",
    "zmax":         r"$Z_{\max}\,[\mathrm{kpc}]$",
    "eccentricity": r"$e$",
    "R_mean":       r"$R_{\mathrm{mean}}\,[\mathrm{kpc}]$",
    "o_h":          r"$[\mathrm{O/H}]$",
    "mg_h":         r"$[\mathrm{Mg/H}]$",
    "si_h":         r"$[\mathrm{Si/H}]$",
    "c_h":          r"$[\mathrm{C/H}]$",
}

In [7]:
# ============================================================
# FEATURE IMPORTANCE FOR ABUNDANCE PREDICTION
# Internal APOGEE + external HARPS
#
# Produces, for each dataset:
#   1. Built-in LightGBM gain importance
#   2. Permutation importance (positive ΔRMSE)
#   3. SHAP global importance (mean |SHAP|)
#   4. Single-feature predictive power (CV RMSE)
#
# Expected use:
#   - use tuned best params for each element
#   - APOGEE train/test split for internal evaluation
#   - train on all APOGEE, evaluate on HARPS for external evaluation
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from copy import deepcopy
from pathlib import Path

from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------
RANDOM_STATE = 42

FEATURES = [
    'U_lsr', 'V_lsr', 'W_lsr', 'P_gal',
    'rapo', 'rperi', 'eccentricity', 'zmax', 'fe_h'
]

PRETTY_LABELS = {
    "U_lsr":        r"$U_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "V_lsr":        r"$V_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "W_lsr":        r"$W_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UV_lsr":       r"$UV_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UW_lsr":       r"$UW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "VW_lsr":       r"$VW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UVW_lsr":      r"$UVW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "abs_U_lsr":    r"$|U|_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "abs_W_lsr":    r"$|W|_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "P_gal":        r"$P_{\mathrm{gal}}\,[\%]$",
    "rapo":         r"$R_{\mathrm{apo}}\,[\mathrm{kpc}]$",
    "rperi":        r"$R_{\mathrm{peri}}\,[\mathrm{kpc}]$",
    "zmax":         r"$Z_{\max}\,[\mathrm{kpc}]$",
    "eccentricity": r"$e$",
    "R_mean":       r"$R_{\mathrm{mean}}\,[\mathrm{kpc}]$",
    "fe_h":         r"$[\mathrm{Fe/H}]$",
    "o_h":          r"$[\mathrm{O/H}]$",
    "mg_h":         r"$[\mathrm{Mg/H}]$",
    "si_h":         r"$[\mathrm{Si/H}]$",
    "c_h":          r"$[\mathrm{C/H}]$",
}

TARGET_LABELS = {
    'o_h': r'$\mathrm{[O/H]}$',
    'mg_h': r'$\mathrm{[Mg/H]}$',
    'si_h': r'$\mathrm{[Si/H]}$',
    'c_h': r'$\mathrm{[C/H]}$'
}

# ------------------------------------------------------------
# PLOTTING STYLE
# ------------------------------------------------------------
plt.style.use('seaborn-v0_8-white')
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 14,
    "figure.dpi": 160
})


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def pretty_feature_names(features, label_map=PRETTY_LABELS):
    return [label_map.get(f, f) for f in features]


def save_series(series, out_csv, label_map=PRETTY_LABELS):
    df = pd.DataFrame({
        "feature": series.index,
        "feature_pretty": [label_map.get(f, f) for f in series.index],
        "value": series.values
    })
    df.to_csv(out_csv, index=False)


def plot_barh(series, title, xlabel, outpath, label_map=PRETTY_LABELS,
              sort_desc=True, top_n=None, color=None):
    s = series.copy()
    if sort_desc:
        s = s.sort_values(ascending=False)
    else:
        s = s.sort_values(ascending=True)

    if top_n is not None:
        s = s.head(top_n)

    s_plot = s.iloc[::-1]
    labels = [label_map.get(f, f) for f in s_plot.index]

    plt.figure(figsize=(10, 7))
    plt.barh(labels, s_plot.values, color=color)
    plt.title(title, weight='bold')
    plt.xlabel(xlabel, weight='bold')
    plt.tight_layout()
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()


# ------------------------------------------------------------
# DATA PREP
# ------------------------------------------------------------
def prepare_element_data(apogee_df, harps_df, element, features=FEATURES):
    cols = features + [element]

    ap = apogee_df[cols].dropna().copy()
    ha = harps_df[cols].dropna().copy()

    X_ap = ap[features].copy()
    y_ap = ap[element].copy()

    X_ha = ha[features].copy()
    y_ha = ha[element].copy()

    return X_ap, y_ap, X_ha, y_ha


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------
def build_lgbm(best_params, random_state=RANDOM_STATE):
    params = deepcopy(best_params)
    params["random_state"] = random_state
    params["verbosity"] = -1
    params["n_jobs"] = -1
    return LGBMRegressor(**params)


# ------------------------------------------------------------
# IMPORTANCE METHODS
# ------------------------------------------------------------
def builtin_importance(model, feature_names):
    if hasattr(model, "booster_"):
        vals = model.booster_.feature_importance(importance_type="gain")
    elif hasattr(model, "feature_importances_"):
        vals = model.feature_importances_
    else:
        raise ValueError("Model does not expose feature importances.")
    return pd.Series(vals, index=feature_names).sort_values(ascending=False)


def permutation_importance_drmse(model, X, y, n_repeats=20, random_state=RANDOM_STATE):
    """
    Returns positive ΔRMSE.
    Larger value = more important.
    """
    perm = permutation_importance(
        model,
        X,
        y,
        scoring="neg_root_mean_squared_error",
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1
    )
    # sklearn returns importance in score space.
    # Since score = -RMSE, a decrease in score corresponds to +RMSE.
    delta_rmse = -perm.importances_mean
    return pd.Series(delta_rmse, index=X.columns).sort_values(ascending=False)


def shap_global_importance(model, X):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    return pd.Series(mean_abs_shap, index=X.columns).sort_values(ascending=False)


def single_feature_cv_rmse(best_params, X, y, cv=5, random_state=RANDOM_STATE):
    """
    One model per single feature using CV.
    Lower RMSE = stronger individual predictive power.
    """
    cv_splitter = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    out = {}

    for feat in X.columns:
        model = build_lgbm(best_params, random_state=random_state)
        scores = cross_val_score(
            model,
            X[[feat]],
            y,
            cv=cv_splitter,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )
        out[feat] = -scores.mean()

    return pd.Series(out).sort_values(ascending=False)


def make_shap_dependence_plot(
    model,
    X,
    feature,
    interaction_feature,
    outpath,
    pretty_labels=PRETTY_LABELS,
    target_label=None
):
    """
    Create a SHAP dependence plot for one feature, colored by another feature.
    """

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    plt.figure(figsize=(8, 6))
    print (pretty_labels.get(feature, feature))

    shap.dependence_plot(
        feature,
        shap_values,
        X,
        interaction_index=interaction_feature,
        show=False
    )

    fig = plt.gcf()
    ax = fig.axes[0]   # main plot axis

    # force labels after SHAP has drawn everything
    ax.set_xlabel(pretty_labels.get(feature, feature))
    if target_label is None:
        ax.set_ylabel("SHAP value")
    else:
        ax.set_ylabel(f"SHAP value for {target_label}")

    # colorbar axis
    if len(fig.axes) > 1:
        cax = fig.axes[-1]
        cax.set_ylabel(pretty_labels.get(interaction_feature, interaction_feature))

    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()


def run_shap_dependence_plots(
    model,
    X,
    outdir,
    dependence_pairs,
    pretty_labels=PRETTY_LABELS,
    target_name=None,
    prefix="LGBM"
):
    """
    dependence_pairs: list of tuples
        e.g. [('fe_h', 'rperi'), ('zmax', 'rapo')]
    """
    ensure_dir(outdir)

    for feature, interaction_feature in dependence_pairs:
        outpath = os.path.join(
            outdir,
            f"{prefix}_dependence_{feature}_by_{interaction_feature}.png"
        )

        make_shap_dependence_plot(
            model=model,
            X=X,
            feature=feature,
            interaction_feature=interaction_feature,
            outpath=outpath,
            pretty_labels=pretty_labels,
            target_label=target_name
        )

# ------------------------------------------------------------
# FULL SUITE FOR ONE TRAIN/EVAL CONFIGURATION
# ------------------------------------------------------------
def run_feature_importance_suite(
    model,
    X_train,
    y_train,
    X_eval,
    y_eval,
    outdir,
    prefix="LGBM",
    label_map=PRETTY_LABELS,
    top_n=None,
    permutation_repeats=20,
    best_params_for_single_feature=None,
    single_feature_X=None,
    single_feature_y=None,
    shap_dependence_X=None,
    shap_dependence_pairs=None,
    target_name=None
):
    ensure_dir(outdir)

    # fit
    fitted = clone(model)
    fitted.fit(X_train, y_train)

    # evaluation RMSE
    y_pred = fitted.predict(X_eval)
    eval_rmse = rmse(y_eval, y_pred)

    # 1. built-in
    bi = builtin_importance(fitted, X_train.columns)
    save_series(bi, os.path.join(outdir, f"{prefix}_builtin_gain.csv"), label_map)
    plot_barh(
        bi,
        title=f"Built-in importance (gain)",
        xlabel="Importance (relative units)",
        outpath=os.path.join(outdir, f"{prefix}_builtin_gain.png"),
        label_map=label_map,
        sort_desc=True,
        top_n=top_n
    )

    # 2. permutation
    perm = permutation_importance_drmse(
        fitted, X_eval, y_eval,
        n_repeats=permutation_repeats,
        random_state=RANDOM_STATE
    )
    save_series(perm, os.path.join(outdir, f"{prefix}_permutation_dRMSE.csv"), label_map)
    plot_barh(
        perm,
        title=f"Permutation importance",
        xlabel=r"$\Delta$RMSE",
        outpath=os.path.join(outdir, f"{prefix}_permutation_dRMSE.png"),
        label_map=label_map,
        sort_desc=True,
        top_n=top_n
    )

    # 3. SHAP
    shap_imp = shap_global_importance(fitted, X_eval)
    save_series(shap_imp, os.path.join(outdir, f"{prefix}_shap_meanabs.csv"), label_map)
    plot_barh(
        shap_imp,
        title=f"SHAP global importance",
        xlabel=r"Mean $|{\rm SHAP}|$",
        outpath=os.path.join(outdir, f"{prefix}_shap_meanabs.png"),
        label_map=label_map,
        sort_desc=True,
        top_n=top_n
    )

    # 5. SHAP dependence plots
    if shap_dependence_X is not None and shap_dependence_pairs is not None:
        dep_dir = os.path.join(outdir, "shap_dependence")
        run_shap_dependence_plots(
            model=fitted,
            X=shap_dependence_X,
            outdir=dep_dir,
            dependence_pairs=shap_dependence_pairs,
            pretty_labels=label_map,
            target_name=None,
            prefix=prefix
        )

    # 4. single-feature predictive power
    if (best_params_for_single_feature is not None) and (single_feature_X is not None) and (single_feature_y is not None):
        sf = single_feature_cv_rmse(
            best_params=best_params_for_single_feature,
            X=single_feature_X,
            y=single_feature_y,
            cv=5,
            random_state=RANDOM_STATE
        )
        save_series(sf, os.path.join(outdir, f"{prefix}_single_feature_rmse.csv"), label_map)
        plot_barh(
            sf,
            title=f"Single-feature predictive power",
            xlabel="RMSE (single-feature CV)",
            outpath=os.path.join(outdir, f"{prefix}_single_feature_rmse.png"),
            label_map=label_map,
            sort_desc=True,   # larger RMSE higher on plot like your example
            top_n=top_n
        )
    else:
        sf = None

    summary = pd.concat([
        bi.rename("builtin_gain"),
        perm.rename("permutation_dRMSE"),
        shap_imp.rename("shap_meanabs"),
        sf.rename("single_feature_cv_rmse") if sf is not None else pd.Series(dtype=float, name="single_feature_cv_rmse")
    ], axis=1)

    summary["builtin_rank"] = summary["builtin_gain"].rank(ascending=False, method="min")
    summary["perm_rank"] = summary["permutation_dRMSE"].rank(ascending=False, method="min")
    summary["shap_rank"] = summary["shap_meanabs"].rank(ascending=False, method="min")
    if sf is not None:
        summary["single_feat_rank"] = summary["single_feature_cv_rmse"].rank(ascending=True, method="min")

    summary.to_csv(os.path.join(outdir, f"{prefix}_summary.csv"))

    return {
        "model": fitted,
        "eval_rmse": eval_rmse,
        "builtin_gain": bi,
        "permutation_dRMSE": perm,
        "shap_meanabs": shap_imp,
        "single_feature_cv_rmse": sf,
        "summary": summary
    }


# ------------------------------------------------------------
# WRAPPER: APOGEE INTERNAL + HARPS EXTERNAL
# ------------------------------------------------------------
def run_apogee_harps_feature_importance(
    apogee_df,
    harps_df,
    element,
    best_params,
    features=FEATURES,
    test_size=0.2,
    random_state=RANDOM_STATE,
    out_root="feature_importance_outputs",
    top_n=None,
    permutation_repeats=20,
    shap_dependence_pairs=None
):
    ensure_dir(out_root)
    out_el = os.path.join(out_root, element)
    ensure_dir(out_el)

    # data
    X_ap, y_ap, X_ha, y_ha = prepare_element_data(apogee_df, harps_df, element, features=features)

    # internal APOGEE split
    X_train_ap, X_test_ap, y_train_ap, y_test_ap = train_test_split(
        X_ap, y_ap,
        test_size=test_size,
        random_state=random_state
    )

    # model with tuned params
    model = build_lgbm(best_params, random_state=random_state)

    # -------------------------------
    # 1) APOGEE internal importance
    # -------------------------------
    apogee_results = run_feature_importance_suite(
        model=model,
        X_train=X_train_ap,
        y_train=y_train_ap,
        X_eval=X_test_ap,
        y_eval=y_test_ap,
        outdir=os.path.join(out_el, "apogee_internal"),
        prefix=f"LGBM_{element}",
        label_map=PRETTY_LABELS,
        top_n=top_n,
        permutation_repeats=permutation_repeats,
        best_params_for_single_feature=best_params,
        single_feature_X=X_ap,
        single_feature_y=y_ap,
        shap_dependence_X=X_ap,
        shap_dependence_pairs=shap_dependence_pairs,
        target_name=element
    )

    # -------------------------------
    # 2) HARPS external importance
    # train on all APOGEE
    # evaluate on HARPS
    # -------------------------------
    harps_results = run_feature_importance_suite(
        model=model,
        X_train=X_ap,
        y_train=y_ap,
        X_eval=X_ha,
        y_eval=y_ha,
        outdir=os.path.join(out_el, "harps_external"),
        prefix="LGBM",
        label_map=PRETTY_LABELS,
        top_n=top_n,
        permutation_repeats=permutation_repeats,
        best_params_for_single_feature=best_params,
        single_feature_X=X_ap,
        single_feature_y=y_ap,
        shap_dependence_X=X_ha,
        shap_dependence_pairs=shap_dependence_pairs,
        target_name=element
    )

    # small text report
    with open(os.path.join(out_el, "report.txt"), "w") as f:
        f.write(f"Element: {element}\n")
        f.write(f"APOGEE size: {len(X_ap)}\n")
        f.write(f"HARPS size: {len(X_ha)}\n")
        f.write(f"APOGEE internal RMSE: {apogee_results['eval_rmse']:.5f}\n")
        f.write(f"HARPS external RMSE: {harps_results['eval_rmse']:.5f}\n")
        f.write("Best params:\n")
        f.write(json.dumps(best_params, indent=2))

    return {
        "X_apogee": X_ap, "y_apogee": y_ap,
        "X_harps": X_ha, "y_harps": y_ha,
        "apogee_internal": apogee_results,
        "harps_external": harps_results
    }


# ------------------------------------------------------------
# LOAD BEST PARAMS
# ------------------------------------------------------------
def load_best_params_from_json(path_json):
    with open(path_json, "r") as f:
        data = json.load(f)
    return data["best_params"]


# ------------------------------------------------------------
# EXAMPLE RUN
# ------------------------------------------------------------
# You already uploaded a JSON for c_h best params. It contains:
# n_estimators=592, learning_rate=0.015266..., num_leaves=19, max_depth=6,
# min_child_samples=162, subsample=0.82327, colsample_bytree=0.87829,
# reg_alpha=4.325e-05, reg_lambda=0.87662 :contentReference[oaicite:0]{index=0}

# Example:
# best_params_c = load_best_params_from_json("/mnt/data/lgbm_best_params_c_h.json")
#
# results_c = run_apogee_harps_feature_importance(
#     apogee_df=apogee,
#     harps_df=harps,
#     element="c_h",
#     best_params=best_params_c,
#     features=FEATURES,
#     test_size=0.2,
#     random_state=42,
#     out_root="feature_importance_outputs",
#     top_n=None,
#     permutation_repeats=20
# )
#
# print(results_c["apogee_internal"]["summary"])
# print(results_c["harps_external"]["summary"])


# ------------------------------------------------------------
# MULTI-ELEMENT RUN
# ------------------------------------------------------------
def run_many_elements(
    apogee_df,
    harps_df,
    elements,
    best_params_paths=None,
    best_params_dict=None,
    features=FEATURES,
    out_root="feature_importance_outputs",
    test_size=0.2,
    random_state=RANDOM_STATE,
    permutation_repeats=20
):
    """
    Supply either:
      - best_params_paths = {'c_h': '/path/to/json', ...}
    or
      - best_params_dict  = {'c_h': {...}, ...}
    """
    all_results = {}

    for el in elements:
        print("\n" + "=" * 70)
        print(f"Running feature importance for {el}")
        print("=" * 70)

        if best_params_dict is not None:
            best_params = best_params_dict[el]
        elif best_params_paths is not None:
            best_params = load_best_params_from_json(best_params_paths[el])
        else:
            raise ValueError("Provide best_params_paths or best_params_dict.")

        res = run_apogee_harps_feature_importance(
            apogee_df=apogee_df,
            harps_df=harps_df,
            element=el,
            best_params=best_params,
            features=features,
            test_size=test_size,
            random_state=random_state,
            out_root=out_root,
            top_n=None,
            permutation_repeats=permutation_repeats
        )

        all_results[el] = res

        print(f"APOGEE internal RMSE ({el}): {res['apogee_internal']['eval_rmse']:.4f}")
        print(f"HARPS external RMSE ({el}): {res['harps_external']['eval_rmse']:.4f}")

        print("\nTop APOGEE built-in:")
        print(res["apogee_internal"]["builtin_gain"].head())

        print("\nTop HARPS permutation:")
        print(res["harps_external"]["permutation_dRMSE"].head())

    return all_results

In [8]:
elements = ['o_h', 'mg_h', 'si_h', 'c_h']

best_params_paths = {
    'o_h':  './models/lgbm_best_params_o_h.json',
    'mg_h': './models/lgbm_best_params_mg_h.json',
    'si_h': './models/lgbm_best_params_si_h.json',
    'c_h':  './models/lgbm_best_params_c_h.json'
}

dependence_pairs = [
    ('fe_h', 'rperi'),
]

all_results = {}

for el in elements:
    best_params = load_best_params_from_json(best_params_paths[el])

    all_results[el] = run_apogee_harps_feature_importance(
        apogee_df=apogee,
        harps_df=harps,
        element=el,
        best_params=best_params,
        features=FEATURES,
        test_size=0.2,
        random_state=42,
        out_root="feature_importance_outputs",
        permutation_repeats=20,
    )

In [9]:
elements = ['o_h', 'mg_h', 'si_h', 'c_h']

best_params_paths = {
    'o_h':  './models/lgbm_best_params_o_h.json',
    'mg_h': './models/lgbm_best_params_mg_h.json',
    'si_h': './models/lgbm_best_params_si_h.json',
    'c_h':  './models/lgbm_best_params_c_h.json'
}
dependence_pairs = [
    ('fe_h', 'rperi'),
    ('fe_h', 'zmax'),
]

for element in elements:
    print(f'Running SHAP dependence plots for {element}')

    best_params = load_best_params_from_json(best_params_paths[element])

    X_ap, y_ap, X_ha, y_ha = prepare_element_data(
        apogee, harps, element, features=FEATURES
    )

    best_lgbm = build_lgbm(best_params)
    best_lgbm.fit(X_ap, y_ap)

    run_shap_dependence_plots(
        model=best_lgbm,
        X=X_ap,   # APOGEE plots
        outdir=f'feature_importance_outputs/{element}/apogee_shap_dependence_only',
        dependence_pairs=dependence_pairs,
        pretty_labels=PRETTY_LABELS,
        target_name=TARGET_LABELS[element],
        prefix=f'LGBM_{element}'
    )

Running SHAP dependence plots for o_h
$[\mathrm{Fe/H}]$
$[\mathrm{Fe/H}]$
Running SHAP dependence plots for mg_h
$[\mathrm{Fe/H}]$
$[\mathrm{Fe/H}]$
Running SHAP dependence plots for si_h
$[\mathrm{Fe/H}]$
$[\mathrm{Fe/H}]$
Running SHAP dependence plots for c_h
$[\mathrm{Fe/H}]$
$[\mathrm{Fe/H}]$


<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

<Figure size 1280x960 with 0 Axes>

In [10]:
elements = ['o_h', 'mg_h', 'si_h', 'c_h']

best_params_paths = {
    'o_h':  './models/lgbm_best_params_o_h.json',
    'mg_h': './models/lgbm_best_params_mg_h.json',
    'si_h': './models/lgbm_best_params_si_h.json',
    'c_h':  './models/lgbm_best_params_c_h.json'
}

all_results = run_many_elements(
    apogee_df=apogee,
    harps_df=harps,
    elements=elements,
    best_params_paths=best_params_paths,
    features=FEATURES,
    out_root='feature_importance_outputs',
    test_size=0.2,
    random_state=42,
    permutation_repeats=20
)


Running feature importance for o_h


KeyboardInterrupt: 